# **Netflix Content Recommendation System**

## **Importing Libraries**

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt
import seaborn as sns

import joblib

## **Loading Dataset**

In [2]:
df = pd.read_csv("Dataset.csv")

df.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


## **Initial Dataset Exploration**

In [3]:
df.shape

(8790, 10)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8790 non-null   object
 1   type          8790 non-null   object
 2   title         8790 non-null   object
 3   director      8790 non-null   object
 4   country       8790 non-null   object
 5   date_added    8790 non-null   object
 6   release_year  8790 non-null   int64 
 7   rating        8790 non-null   object
 8   duration      8790 non-null   object
 9   listed_in     8790 non-null   object
dtypes: int64(1), object(9)
memory usage: 686.8+ KB


## **Relevant Columns**

In [5]:
content_columns = [
    "type",
    "title",
    "director",
    "country",
    "release_year",
    "rating",
    "duration",
    "listed_in"
]

df[content_columns].head()

,type,title,director,country,release_year,rating,duration,listed_in
0,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2020,PG-13,90 min,Documentaries
1,TV Show,Ganglands,Julien Leclercq,France,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,TV Show,Midnight Mass,Mike Flanagan,United States,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,Movie,Sankofa,Haile Gerima,United States,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


## **Missing Values**

In [6]:
df[content_columns].isnull().sum()

,0
type,0
title,0
director,0
country,0
release_year,0
rating,0
duration,0
listed_in,0


In [7]:
df["type"].value_counts()

,count
type,
Movie,6126
TV Show,2664


In [8]:
df["listed_in"].value_counts().head(10)

,count
listed_in,
"Dramas, International Movies",362
Documentaries,359
Stand-Up Comedy,334
"Comedies, Dramas, International Movies",274
"Dramas, Independent Movies, International Movies",252
Kids' TV,219
Children & Family Movies,215
"Children & Family Movies, Comedies",201
"Documentaries, International Movies",186


## **Prepare Content Features**

### **Clean the Duration Feature**

In [9]:
df["duration"] = (
    df["duration"]
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

### **Combine Content Features**

In [10]:
df["content"] = (
    df["listed_in"] + " " +
    df["type"] + " " +
    df["rating"] + " " +
    df["duration"]
)

In [11]:
df[["title", "listed_in", "type", "rating", "duration", "content"]].head()

,title,listed_in,type,rating,duration,content
0,Dick Johnson Is Dead,Documentaries,Movie,PG-13,90_min,Documentaries Movie PG-13 90_min
1,Ganglands,"Crime TV Shows, International TV Shows, TV Act...",TV Show,TV-MA,1_season,"Crime TV Shows, International TV Shows, TV Act..."
2,Midnight Mass,"TV Dramas, TV Horror, TV Mysteries",TV Show,TV-MA,1_season,"TV Dramas, TV Horror, TV Mysteries TV Show TV-..."
3,Confessions of an Invisible Girl,"Children & Family Movies, Comedies",Movie,TV-PG,91_min,"Children & Family Movies, Comedies Movie TV-PG..."
4,Sankofa,"Dramas, Independent Movies, International Movies",Movie,TV-MA,125_min,"Dramas, Independent Movies, International Movi..."


### **Check the Prepared Features**

In [12]:
df["content"].isnull().sum()

np.int64(0)

In [13]:
df[["title", "content"]].sample(5, random_state=42)

,title,content
3942,Kiss & Cry,"Dramas, International Movies, Romantic Movies ..."
8360,Beating Again,"International TV Shows, Korean TV Shows, Roman..."
221,Running Man,"Kids' TV, TV Comedies TV Show TV-Y7 1_season"
4883,Farce,"Comedies, International Movies Movie TV-MA 94_min"
2210,Pretty Little Stalker,Thrillers Movie TV-14 84_min


## **Convert Text into TF-IDF Features**

### **Create the TF-IDF Matrix**

In [14]:
tfidf = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(df["content"])

tfidf_matrix.shape

(8790, 275)

### **View the TF-IDF Features**

In [15]:
tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 59195 stored elements and shape (8790, 275)>

## **Calculate Content Similarity**

### **Calculate Cosine Similarity**

In [16]:
cosine_sim = cosine_similarity(tfidf_matrix)

cosine_sim.shape

(8790, 8790)

### **Create a Title Index**

In [17]:
indices = pd.Series(df.index, index=df["title"]).drop_duplicates()

In [18]:
indices.head()

,0
title,
Dick Johnson Is Dead,0
Ganglands,1
Midnight Mass,2
Confessions of an Invisible Girl,3
Sankofa,4


## **Generate Recommendations**

### **Recommendation Function**

In [19]:
title_lookup = pd.Series(
    df.index,
    index=df["title"].str.lower()
).drop_duplicates()

In [20]:
def recommend(title, n=5):
    idx = title_lookup.get(title.lower())

    if idx is None:
        return pd.DataFrame()

    similarity_scores = list(enumerate(cosine_sim[idx]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:n + 1]

    movie_indices = [i[0] for i in similarity_scores]

    recommendations = df.loc[
        movie_indices,
        ["title", "type", "listed_in", "rating", "duration"]
    ].copy()

    recommendations["similarity_score"] = [
        round(score, 3) for _, score in similarity_scores
    ]

    return recommendations.reset_index(drop=True)

### **Test the Recommendation System**

In [21]:
recommend("enola holmes", n=5)

,title,type,listed_in,rating,duration,similarity_score
0,The Sum of All Fears,Movie,Action & Adventure,PG-13,124_min,0.843
1,XXx,Movie,"Action & Adventure, Sports Movies",PG-13,124_min,0.764
2,Freedom Writers,Movie,Dramas,PG-13,124_min,0.757
3,The Twilight Saga: Eclipse,Movie,"Dramas, Romantic Movies",PG-13,124_min,0.721
4,Loving,Movie,"Dramas, Romantic Movies",PG-13,124_min,0.721


In [22]:
recommend("locke & key", n=5)

,title,type,listed_in,rating,duration,similarity_score
0,Hit & Run,TV Show,"TV Action & Adventure, TV Dramas, TV Mysteries",TV-MA,1_season,0.952
1,The Untamed,TV Show,"International TV Shows, TV Action & Adventure,...",TV-14,1_season,0.935
2,Alice in Borderland,TV Show,"International TV Shows, TV Action & Adventure,...",TV-MA,1_season,0.888
3,The Ghost Bride,TV Show,"International TV Shows, TV Dramas, TV Mysteries",TV-14,1_season,0.850
4,More to Say,TV Show,"International TV Shows, TV Dramas, TV Mysteries",TV-14,1_season,0.850


## **Evaluate Recommendation Quality**

### **Helper Function for Genre Overlap**

In [23]:
def genre_overlap(title, recommendations):
    original = df.loc[
        df["title"].str.lower() == title.lower(),
        "listed_in"
    ]

    if original.empty or recommendations.empty:
        return 0

    original_genres = set(
        genre.strip().lower()
        for genre in original.iloc[0].split(",")
    )

    matches = 0

    for genres in recommendations["listed_in"]:
        recommended_genres = set(
            genre.strip().lower()
            for genre in genres.split(",")
        )

        if original_genres.intersection(recommended_genres):
            matches += 1

    return round((matches / len(recommendations)) * 100, 2)

### **Evaluate a Few Titles**

In [24]:
test_titles = [
    "Dick Johnson Is Dead",
    "Ganglands",
    "enola holmes",
    "locke & key"
]

In [25]:
for title in test_titles:
    recommendations = recommend(title, n=5)

    print(f"\n{title}")
    print("-" * len(title))
    display(recommendations)

    overlap = genre_overlap(title, recommendations)
    print(f"Genre overlap: {overlap}%")


Dick Johnson Is Dead
--------------------


,title,type,listed_in,rating,duration,similarity_score
0,Misha and the Wolves,Movie,"Documentaries, International Movies",PG-13,90_min,0.955
1,Inequality for All,Movie,Documentaries,PG,90_min,0.867
2,Austin Powers: International Man of Mystery,Movie,Comedies,PG-13,90_min,0.858
3,Little Nicky,Movie,Comedies,PG-13,90_min,0.858
4,Cowspiracy: The Sustainability Secret,Movie,Documentaries,TV-PG,90_min,0.853


Genre overlap: 60.0%

Ganglands
---------


,title,type,listed_in,rating,duration,similarity_score
0,Bangkok Breaking,TV Show,"Crime TV Shows, International TV Shows, TV Act...",TV-MA,1_season,1.0
1,Fatal Destiny,TV Show,"Crime TV Shows, International TV Shows, TV Act...",TV-MA,1_season,1.0
2,Dealer,TV Show,"Crime TV Shows, International TV Shows, TV Act...",TV-MA,1_season,1.0
3,Nowhere Man,TV Show,"Crime TV Shows, International TV Shows, TV Act...",TV-MA,1_season,1.0
4,Monkey Twins,TV Show,"Crime TV Shows, International TV Shows, TV Act...",TV-MA,1_season,1.0


Genre overlap: 100.0%

enola holmes
------------


,title,type,listed_in,rating,duration,similarity_score
0,The Sum of All Fears,Movie,Action & Adventure,PG-13,124_min,0.843
1,XXx,Movie,"Action & Adventure, Sports Movies",PG-13,124_min,0.764
2,Freedom Writers,Movie,Dramas,PG-13,124_min,0.757
3,The Twilight Saga: Eclipse,Movie,"Dramas, Romantic Movies",PG-13,124_min,0.721
4,Loving,Movie,"Dramas, Romantic Movies",PG-13,124_min,0.721


Genre overlap: 100.0%

locke & key
-----------


,title,type,listed_in,rating,duration,similarity_score
0,Hit & Run,TV Show,"TV Action & Adventure, TV Dramas, TV Mysteries",TV-MA,1_season,0.952
1,The Untamed,TV Show,"International TV Shows, TV Action & Adventure,...",TV-14,1_season,0.935
2,Alice in Borderland,TV Show,"International TV Shows, TV Action & Adventure,...",TV-MA,1_season,0.888
3,The Ghost Bride,TV Show,"International TV Shows, TV Dramas, TV Mysteries",TV-14,1_season,0.850
4,More to Say,TV Show,"International TV Shows, TV Dramas, TV Mysteries",TV-14,1_season,0.850


Genre overlap: 100.0%


## **Saving Model Artifacts**

In [26]:
joblib.dump(tfidf, "tfidf_vectorizer.pkl")
joblib.dump(cosine_sim, "cosine_similarity.pkl")
joblib.dump(title_lookup, "title_lookup.pkl")

['title_lookup.pkl']

## **Saving the Dataset Used by the App**

In [27]:
app_df = df[
    ["title", "type", "listed_in", "rating", "duration"]
].copy()

app_df.to_csv("netflix_recommendation_data.csv", index=False)